[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BMGLab/BFB/blob/main/W06_Variation__independence_and_multiple_testing.ipynb)

# Week 06 | Variation, independence and multiple testing

**Core practical: 45 minutes.** Use a known null simulation to distinguish false positives from posterior probabilities.

No paid AI tool, local installation or external dataset download is required. Open it in Colab with the badge above, then **File > Save a copy in Drive** before you start so your work is kept. Run the cells from top to bottom. Local Jupyter with Python 3.9+ is an alternative. Plotting is optional.

**Data boundary:** Every generated value is synthetic. Explicit REAL EXCERPT / REAL record cards are separately labelled and limited to their stated purpose. No patient data should be entered.

## Before running (5 min)
Technical repeats share biological origins; more observations are not automatically more independent evidence.

Write a prediction in the response cell before executing the analysis.

## Setup (5 min)
Replace only `COURSE_ID` with your assigned pseudonym. A seed supports reproducibility; it is not proof of authorship.

In [ ]:
import hashlib, json, math, random, statistics, sys
from pathlib import Path
COURSE_ID = "demo-001"  # Replace with your assigned course pseudonym, not your name or national ID.
SEED = int(hashlib.sha256(COURSE_ID.encode()).hexdigest()[:8], 16)
rng = random.Random(SEED)
RESULTS = {}
print("Python", sys.version.split()[0], "| course ID", COURSE_ID, "| seed", SEED)

def mean(values):
    if not values: raise ValueError("Cannot average an empty list")
    return sum(values) / len(values)

def bh_adjust(pvalues):
    """Benjamini-Hochberg adjusted p-values, returned in original order."""
    if any(not 0 <= p <= 1 for p in pvalues): raise ValueError("p must be in [0,1]")
    m = len(pvalues)
    order = sorted(range(m), key=lambda i: pvalues[i])
    out = [0.0] * m
    running = 1.0
    for j in range(m - 1, -1, -1):
        i = order[j]
        running = min(running, pvalues[i] * m / (j + 1))
        out[i] = running
    return out

def optional_plot(labels, values, ylabel, title):
    """Plot when matplotlib is available; numerical work never requires it."""
    try:
        import matplotlib.pyplot as plt
    except ImportError:
        print("Plot unavailable; use the numerical table above.")
        return
    fig, ax = plt.subplots(figsize=(7, 4))
    ax.bar(labels, values)
    ax.set(ylabel=ylabel, title=title)
    fig.tight_layout()
    plt.show()

### Prediction
Write your prediction here before running the investigation, then copy it into PREDICTION in the response cell.

## Guided investigation (20 min)
1. Adjust a four-test example by hand.
2. Run 500 all-null families of 100 uniform p-values.
3. Compare unadjusted counts with BH discoveries.
4. Explain the family-level interpretation rather than claiming every run contains exactly five errors.

In [ ]:
p = [0.001, 0.010, 0.040, 0.200]
adjusted = bh_adjust(p)
print("Worked example:", list(zip(p, adjusted)))
assert all(abs(a-b) < 1e-10 for a,b in zip(adjusted,[0.004,0.020,0.0533333333333,0.2]))
trials, m, alpha = 500, 100, 0.05
raw_counts, bh_counts = [], []
for _ in range(trials):
    family = [rng.random() for _ in range(m)]  # Valid uniform p-values under this simulated global null.
    raw_counts.append(sum(x <= alpha for x in family))
    bh_counts.append(sum(x <= alpha for x in bh_adjust(family)))
RESULTS = {"data_status": "SYNTHETIC all-null p-values; independent Uniform(0,1)",
           "worked_BH": adjusted, "mean_raw_false_positives": mean(raw_counts),
           "fraction_families_with_BH_discovery": mean([n > 0 for n in bh_counts]),
           "mean_BH_discoveries": mean(bh_counts)}
print(json.dumps(RESULTS, indent=2))
optional_plot(["Unadjusted", "BH"], [mean(raw_counts), mean(bh_counts)],
              "Mean false discoveries / family", "Simulation only: every hypothesis is null")

## Explain the evidence (10 min)
**Q1.** Which of the four worked-example tests pass BH at 0.05? Show adjusted values.

**Q2.** Why is the expected number of unadjusted false positives 100 x 0.05, but not necessarily the observed number in a run?

**Q3.** Under this global null, why is FDR the probability of any discovery? Why is this not an individual gene probability?

In [ ]:
PREDICTION = ""  # Fill before the analysis.
RESPONSES = {"Q1": "", "Q2": "", "Q3": ""}
AI_DISCLOSURE = "No AI used."  # Change to tool, date, purpose, and checks if you used one.
CHECK_PERFORMED = ""  # Describe one actual check, even if it found no error.

## Save and submit (5 min)
Complete your responses above, restart the runtime and run all. Download the `.ipynb` and generated summary JSON. In Colab the JSON is in the Files sidebar. Upload both to the course LMS assignment. Do not email patient data. A completion flag checks presence of responses, not scientific correctness. Paper fallback may be submitted as a legible scan with the same answers.

In [ ]:
required = [PREDICTION, CHECK_PERFORMED] + list(RESPONSES.values())
complete = all(isinstance(x,str) and x.strip() for x in required)
safe_id = "".join(c for c in COURSE_ID if c.isalnum() or c in "-_")[:40] or "anonymous"
report = {"week":6, "course_id":COURSE_ID,"seed":SEED,"python":sys.version.split()[0],
          "prediction":PREDICTION,"results":RESULTS,"responses":RESPONSES,
          "check_performed":CHECK_PERFORMED,"AI_disclosure":AI_DISCLOSURE,
          "response_fields_complete":complete}
output = Path(f"W06_{safe_id}_summary.json")
output.write_text(json.dumps(report,indent=2),encoding="utf-8")
print("Saved:",output)
print("Ready for review" if complete else "DRAFT: fill prediction, responses and check before submission")

## Paper / device-free route
Use the four p-values and multiply ordered values by 4/rank, then impose reverse cumulative minima. For 100 tests, expected unadjusted errors = 5.

## Optional extension
Optional: show why dependence affects a multiple-testing guarantee; do not memorize a proof.

## Sources
- [S07] American Statistical Association. Statement on statistical significance and p-values (2016). https://www.amstat.org/asa/files/pdfs/P-ValueStatement.pdf
- [S08] Benjamini and Hochberg (1995). Controlling the false discovery rate: a practical and powerful approach to multiple testing. https://doi.org/10.1111/j.2517-6161.1995.tb02031.x
- [S10] DESeq2: Analyzing RNA-seq data with DESeq2, release vignette. https://bioconductor.org/packages/release/bioc/vignettes/DESeq2/inst/doc/DESeq2.html
- [S12] Squair et al. (2021). Confronting false discoveries in single-cell differential expression. https://doi.org/10.1038/s41467-021-25960-2